# Physics-Informed Neural Network (PINN) for 2D Navier-Stokes

This notebook implements a PINN to solve the **2D Incompressible Navier-Stokes Equations** for a free-evolving flow field. We use the **Taylor-Green Vortex** as a benchmark problem.

### 1. The Governing Equations
For an incompressible Newtonian fluid without external forces, the governing equations are:

**1. Continuity Equation (Mass Conservation):**
$$ \nabla \cdot \mathbf{u} = 0 \implies \frac{\partial u}{\partial x} + \frac{\partial v}{\partial y} = 0 $$

**2. Momentum Equations (Navier-Stokes):**
$$ \frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} + v\frac{\partial u}{\partial y} = -\frac{\partial p}{\partial x} + \nu \left( \frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2} \right) $$
$$ \frac{\partial v}{\partial t} + u\frac{\partial v}{\partial x} + v\frac{\partial v}{\partial y} = -\frac{\partial p}{\partial y} + \nu \left( \frac{\partial^2 v}{\partial x^2} + \frac{\partial^2 v}{\partial y^2} \right) $$

Where $u, v$ are the velocity components, $p$ is the kinematic pressure, and $\nu$ is the kinematic viscosity.

### 2. The Taylor-Green Vortex (Analytical Solution)
We define the domain $x, y \in [0, 2\pi]$ and $t \in [0, 1]$. With $\nu = 0.01$, the exact solution is:
* $u(x,y,t) = \sin(x)\cos(y)e^{-2\nu t}$
* $v(x,y,t) = -\cos(x)\sin(y)e^{-2\nu t}$
* $p(x,y,t) = \frac{1}{4}(\cos(2x) + \cos(2y))e^{-4\nu t}$

In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# Seed per riproducibilità
torch.manual_seed(42)
np.random.seed(42)

# ==========================================
# SELEZIONE DEL DEVICE (CPU vs GPU)
# ==========================================
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"🚀 GPU Trovata! PyTorch userà: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')
    print("⚠️ Nessuna GPU trovata, PyTorch userà la CPU.")

⚠️ Nessuna GPU trovata, PyTorch userà la CPU.


### Neural Network Architecture
The network takes 3 inputs $(x, y, t)$ and predicts 3 outputs $(u, v, p)$. We use a deeper MLP with 5 hidden layers (50 neurons each) and `Tanh` activations.

In [2]:
class PINN_NS(nn.Module):
    def __init__(self):
        super(PINN_NS, self).__init__()
        # Input: [x, y, t] (3) -> Output: [u, v, p] (3)
        self.net = nn.Sequential(
            nn.Linear(3, 50), nn.Tanh(),
            nn.Linear(50, 50), nn.Tanh(),
            nn.Linear(50, 50), nn.Tanh(),
            nn.Linear(50, 50), nn.Tanh(),
            nn.Linear(50, 50), nn.Tanh(),
            nn.Linear(50, 3)
        )

    def forward(self, x, y, t):
        inputs = torch.cat([x, y, t], dim=1)
        return self.net(inputs)

In [3]:
def navier_stokes_loss(model, x, y, t, nu=0.01):
    x.requires_grad_(True)
    y.requires_grad_(True)
    t.requires_grad_(True)
    
    # Previsione: u, v, p
    preds = model(x, y, t)
    u, v, p = preds[:, 0:1], preds[:, 1:2], preds[:, 2:3]
    
    # Derivate prime
    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_y = torch.autograd.grad(u, y, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    
    v_t = torch.autograd.grad(v, t, grad_outputs=torch.ones_like(v), create_graph=True)[0]
    v_x = torch.autograd.grad(v, x, grad_outputs=torch.ones_like(v), create_graph=True)[0]
    v_y = torch.autograd.grad(v, y, grad_outputs=torch.ones_like(v), create_graph=True)[0]
    
    p_x = torch.autograd.grad(p, x, grad_outputs=torch.ones_like(p), create_graph=True)[0]
    p_y = torch.autograd.grad(p, y, grad_outputs=torch.ones_like(p), create_graph=True)[0]
    
    # Derivate seconde
    u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    u_yy = torch.autograd.grad(u_y, y, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]
    v_xx = torch.autograd.grad(v_x, x, grad_outputs=torch.ones_like(v_x), create_graph=True)[0]
    v_yy = torch.autograd.grad(v_y, y, grad_outputs=torch.ones_like(v_y), create_graph=True)[0]
    
    # 1. Equazione di Continuità (Massa)
    e_c = u_x + v_y
    
    # 2. Equazioni di Quantità di Moto (Momentum X e Y)
    e_u = u_t + u * u_x + v * u_y + p_x - nu * (u_xx + u_yy)
    e_v = v_t + u * v_x + v * v_y + p_y - nu * (v_xx + v_yy)
    
    # La loss totale della PDE è la media dei quadrati dei tre residui fisici
    pde_loss = torch.mean(e_c**2) + torch.mean(e_u**2) + torch.mean(e_v**2)
    return pde_loss

In [4]:
# Dominio: x,y in [0, 2*pi], t in [0, 1]
N_pde = 10000  
N_bc = 2000    
N_ic = 2000    
nu = 0.01

def tg_u(x, y, t): return torch.sin(x) * torch.cos(y) * torch.exp(-2 * nu * t)
def tg_v(x, y, t): return -torch.cos(x) * torch.sin(y) * torch.exp(-2 * nu * t)
def tg_p(x, y, t): return 0.25 * (torch.cos(2*x) + torch.cos(2*y)) * torch.exp(-4 * nu * t)


x_pde = (torch.rand(N_pde, 1) * 2 * np.pi).to(device)
y_pde = (torch.rand(N_pde, 1) * 2 * np.pi).to(device)
t_pde = torch.rand(N_pde, 1).to(device)

# Condizioni Iniziali (t=0)
x_ic = (torch.rand(N_ic, 1) * 2 * np.pi).to(device)
y_ic = (torch.rand(N_ic, 1) * 2 * np.pi).to(device)
t_ic = torch.zeros(N_ic, 1).to(device)
u_ic_true, v_ic_true, p_ic_true = tg_u(x_ic, y_ic, t_ic), tg_v(x_ic, y_ic, t_ic), tg_p(x_ic, y_ic, t_ic)

# Condizioni al Contorno (Bordi random)
x_bc = torch.cat([torch.zeros(N_bc//4, 1), torch.ones(N_bc//4, 1)*2*np.pi, torch.rand(N_bc//2, 1)*2*np.pi], dim=0).to(device)
y_bc = torch.cat([torch.rand(N_bc//2, 1)*2*np.pi, torch.zeros(N_bc//4, 1), torch.ones(N_bc//4, 1)*2*np.pi], dim=0).to(device)
t_bc = torch.rand(N_bc, 1).to(device)
u_bc_true, v_bc_true = tg_u(x_bc, y_bc, t_bc), tg_v(x_bc, y_bc, t_bc)

In [5]:

model = PINN_NS().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)

epochs = 3000 
for epoch in range(epochs):
    optimizer.zero_grad()
    
    loss_pde = navier_stokes_loss(model, x_pde, y_pde, t_pde, nu=nu)
    
    preds_ic = model(x_ic, y_ic, t_ic)
    loss_ic = torch.mean((preds_ic[:, 0:1] - u_ic_true)**2) + \
              torch.mean((preds_ic[:, 1:2] - v_ic_true)**2) + \
              torch.mean((preds_ic[:, 2:3] - p_ic_true)**2)
              
    preds_bc = model(x_bc, y_bc, t_bc)
    loss_bc = torch.mean((preds_bc[:, 0:1] - u_bc_true)**2) + \
              torch.mean((preds_bc[:, 1:2] - v_bc_true)**2)
              
    loss = loss_pde + loss_ic + loss_bc
    loss.backward()
    optimizer.step()
    
    if epoch % 500 == 0:
        print(f"Epoch {epoch:4d} | Total Loss: {loss.item():.5f} (PDE: {loss_pde.item():.5f})")

print("Training completato su GPU!")

Epoch    0 | Total Loss: 1.13516 (PDE: 0.00008)
Epoch  500 | Total Loss: 0.29960 (PDE: 0.05273)
Epoch 1000 | Total Loss: 0.09196 (PDE: 0.02455)
Epoch 1500 | Total Loss: 0.02110 (PDE: 0.01122)


KeyboardInterrupt: 

In [ ]:
# Griglia spaziale
x_test = torch.linspace(0, 2*np.pi, 50)
y_test = torch.linspace(0, 2*np.pi, 50)
X, Y = torch.meshgrid(x_test, y_test, indexing='ij')

X_flat_gpu = X.reshape(-1, 1).to(device)
Y_flat_gpu = Y.reshape(-1, 1).to(device)

fig, ax = plt.subplots(figsize=(6, 5))
cax = ax.contourf(X.numpy(), Y.numpy(), np.zeros_like(X.numpy()), 50, cmap='viridis')
fig.colorbar(cax, ax=ax, label='Velocity Magnitude $\sqrt{u^2 + v^2}$')
ax.set_title("Taylor-Green Vortex - Time: 0.0s")
ax.set_xlabel('x')
ax.set_ylabel('y')

def update(frame):
    t_val = frame / 20.0 
    T_flat_gpu = (torch.ones_like(X_flat_gpu) * t_val).to(device)
    
    with torch.no_grad():
        preds = model(X_flat_gpu, Y_flat_gpu, T_flat_gpu)
        # Riportiamo u e v sulla CPU per disegnarli!
        u_pred = preds[:, 0].detach().cpu().reshape(50, 50).numpy()
        v_pred = preds[:, 1].detach().cpu().reshape(50, 50).numpy()
        
    vel_mag = np.sqrt(u_pred**2 + v_pred**2)
    
    ax.clear()
    cax = ax.contourf(X.numpy(), Y.numpy(), vel_mag, 50, cmap='viridis', vmin=0, vmax=1.0)
    ax.set_title(f"PINN Navier-Stokes | Velocity | t = {t_val:.2f}s")
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    return cax,

print("Generazione dell'animazione in corso...")
ani = animation.FuncAnimation(fig, update, frames=20, interval=200, blit=False)
ani.save('navier_stokes_pinn.gif', writer='pillow', fps=5)
print("Animazione salvata come 'navier_stokes_pinn.gif'!")

HTML(ani.to_jshtml())